# LAB 08 — Tokenization (Bài 10 và 12)

Chạy theo thứ tự trong môi trường đã cài `requirements.txt`. Trên Colab, clone repo và `%cd /content/NLP/Lab8` trước (xem README). Các token/IDs bên dưới lấy từ tokenizer thật. Phần tự giải thích nằm trong `calculations.md`.

## 1. Imports và môi trường

In [1]:
from pathlib import Path
import sys
import json
import gc
import pandas as pd
import torch
from IPython.display import display, Image

candidates = []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "Lab8", parent / "NLP" / "Lab8", parent / "lab08"])
LAB_DIR = next((p.resolve() for p in candidates
                if (p / "pretrained_classification.py").is_file()), None)
if LAB_DIR is None:
    raise FileNotFoundError("Hãy clone/upload repo NLP rồi mở notebook trong Lab8; xem README.md.")
sys.path.insert(0, str(LAB_DIR))
from pretrained_classification import (
    Config, LabRunner, build_model, count_parameters, environment_info,
    infer_texts, load_tokenizer, set_seed,
)
display(pd.Series(environment_info(), name="Environment").to_frame())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(4)
set_seed(42)
print("Device:", DEVICE)


/home/nguyenducthang/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Environment
python,3.12.3
torch,2.12.1+cpu
transformers,5.8.1
datasets,4.8.4
accelerate,1.13.0
scikit-learn,1.8.0
numpy,2.3.5
pandas,3.0.1
matplotlib,3.10.9
nbformat,5.11.1


Device: cpu


## 2. Load tokenizer và configuration

In [2]:
MODEL_NAME = "distilbert/distilbert-base-uncased"
# tokenizer tải tự động nếu chưa có trong cache.
tokenizer = load_tokenizer(MODEL_NAME)
print("Model:", MODEL_NAME)
print("Tokenizer:", type(tokenizer).__name__, "| fast:", tokenizer.is_fast)
print("Vocabulary:", len(tokenizer), "| model_max_length:", tokenizer.model_max_length)
print("Padding side:", tokenizer.padding_side, "| truncation side:", tokenizer.truncation_side)
print("Special tokens:", tokenizer.special_tokens_map)


Model: distilbert/distilbert-base-uncased
Tokenizer: BertTokenizer | fast: True
Vocabulary: 30522 | model_max_length: 512
Padding side: right | truncation side: right
Special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


## 3. Bài 10 — Một câu và word-to-subword mapping

In [3]:
text = "The cats are playing."
encoding = tokenizer(text, return_special_tokens_mask=True)
tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
print("Text:", text)
print("Keys:", encoding.keys())
print("Token sequence:", tokens)
print("Token IDs:", encoding["input_ids"])
print("Attention mask:", encoding["attention_mask"])
print("word_ids (raw tokenizer units):", encoding.word_ids() if tokenizer.is_fast else "unsupported")
display(pd.DataFrame({"token": tokens, "id": encoding["input_ids"],
                      "special": encoding["special_tokens_mask"],
                      "attention_mask": encoding["attention_mask"]}))

words = text.split()
# dùng whitespace words để dấu chấm vẫn thuộc từ playing. trong mapping này.
aligned = tokenizer(words, is_split_into_words=True)
word_ids = aligned.word_ids() if tokenizer.is_fast else [None] * len(aligned["input_ids"])
display(pd.DataFrame({"token":tokenizer.convert_ids_to_tokens(aligned["input_ids"]),
                      "word_id":word_ids,
                      "source_word":[words[i] if i is not None else "SPECIAL" for i in word_ids]}))
# content pieces gồm dấu câu; ## chỉ đếm continuation pieces của WordPiece.
counts = {"whitespace_words":len(words), "total_tokens_including_special":len(tokens),
          "special_tokens":sum(encoding["special_tokens_mask"]),
          "content_token_pieces":sum(v == 0 for v in encoding["special_tokens_mask"]),
          "wordpiece_continuations_##":sum(t.startswith("##") for t in tokens)}
display(pd.Series(counts, name="Count").to_frame())


Text: The cats are playing.
Keys: KeysView({'input_ids': [101, 1996, 8870, 2024, 2652, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1], 'special_tokens_mask': [1, 0, 0, 0, 0, 0, 1]})
Token sequence: ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]']
Token IDs: [101, 1996, 8870, 2024, 2652, 1012, 102]
Attention mask: [1, 1, 1, 1, 1, 1, 1]
word_ids (raw tokenizer units): [None, 0, 1, 2, 3, 4, None]


,token,id,special,attention_mask
0,[CLS],101,1,1
1,the,1996,0,1
2,cats,8870,0,1
3,are,2024,0,1
4,playing,2652,0,1
5,.,1012,0,1
6,[SEP],102,1,1


,token,word_id,source_word
0,[CLS],NaN,SPECIAL
1,the,0.0,The
2,cats,1.0,cats
3,are,2.0,are
4,playing,3.0,playing.
5,.,3.0,playing.
6,[SEP],NaN,SPECIAL


,Count
whitespace_words,4
total_tokens_including_special,7
special_tokens,2
content_token_pieces,5
wordpiece_continuations_##,0


`content_token_pieces` bao gồm cả dấu câu; số mảnh này khác số whitespace words. Cột `word_id` là output thật của fast tokenizer. `TODO(student)`: tự ghi lại và giải thích kết quả trong calculations.md.

In [4]:
subword_examples = ["unbelievably", "tokenization", "antidisestablishmentarianism"]
for word in subword_examples:
    ids = tokenizer(word, add_special_tokens=False)["input_ids"]
    print(word, "→", tokenizer.convert_ids_to_tokens(ids), "→", ids)

unbelievably → ['un', '##bel', '##ie', '##va', '##bly'] → [4895, 8671, 2666, 3567, 6321]
tokenization → ['token', '##ization'] → [19204, 3989]
antidisestablishmentarianism → ['anti', '##dis', '##est', '##ab', '##lish', '##ment', '##arian', '##ism'] → [3424, 10521, 4355, 7875, 13602, 3672, 12199, 2964]


## 4. Batch có padding và attention mask

In [5]:
texts = ["The cats are playing.", "Bad.", "This surprisingly unusual movie is wonderful."]
batch = tokenizer(texts, padding=True, truncation=True, max_length=32, return_tensors="pt")
print("input_ids:", batch["input_ids"])
print("attention_mask:", batch["attention_mask"])
print("Shapes:", batch["input_ids"].shape, batch["attention_mask"].shape)
assert batch["input_ids"].shape == batch["attention_mask"].shape
assert torch.equal(batch["input_ids"] == tokenizer.pad_token_id, batch["attention_mask"] == 0)
assert ((batch["input_ids"] >= 0) & (batch["input_ids"] < len(tokenizer))).all()
for i, sentence in enumerate(texts):
    print(sentence)
    print(list(zip(tokenizer.convert_ids_to_tokens(batch["input_ids"][i]), batch["attention_mask"][i].tolist())))


input_ids: tensor([[  101,  1996,  8870,  2024,  2652,  1012,   102,     0,     0],
        [  101,  2919,  1012,   102,     0,     0,     0,     0,     0],
        [  101,  2023, 10889,  5866,  3185,  2003,  6919,  1012,   102]])
attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])
Shapes: torch.Size([3, 9]) torch.Size([3, 9])
The cats are playing.
[('[CLS]', 1), ('the', 1), ('cats', 1), ('are', 1), ('playing', 1), ('.', 1), ('[SEP]', 1), ('[PAD]', 0), ('[PAD]', 0)]
Bad.
[('[CLS]', 1), ('bad', 1), ('.', 1), ('[SEP]', 1), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0)]
This surprisingly unusual movie is wonderful.
[('[CLS]', 1), ('this', 1), ('surprisingly', 1), ('unusual', 1), ('movie', 1), ('is', 1), ('wonderful', 1), ('.', 1), ('[SEP]', 1)]


## 5. Decode IDs về văn bản

In [6]:
for ids in batch["input_ids"]:
    print("With special tokens:", tokenizer.decode(ids, skip_special_tokens=False))
    print("Without special tokens:", tokenizer.decode(ids, skip_special_tokens=True))


With special tokens: [CLS] the cats are playing. [SEP] [PAD] [PAD]
Without special tokens: the cats are playing.
With special tokens: [CLS] bad. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD]
Without special tokens: bad.
With special tokens: [CLS] this surprisingly unusual movie is wonderful. [SEP]
Without special tokens: this surprisingly unusual movie is wonderful.


## 6. Truncation và maximum sequence length

In [7]:
long_text = "The movie has beautiful scenes but an unconvincing ending. " * 20
original_pieces = len(tokenizer(long_text, add_special_tokens=False)["input_ids"])
rows = []
for length in [8, 16, 32, 64, 128, 256]:
    encoded = tokenizer(long_text, truncation=True, max_length=length,
                        return_special_tokens_mask=True)
    kept = sum(v == 0 for v in encoded["special_tokens_mask"])
    rows.append({"max_length":length, "original_content_pieces":original_pieces,
                 "retained_content_pieces":kept, "total_tokens":len(encoded["input_ids"]),
                 "decoded":tokenizer.decode(encoded["input_ids"],skip_special_tokens=True)})
    assert len(encoded["input_ids"]) <= length
display(pd.DataFrame(rows))


,max_length,original_content_pieces,retained_content_pieces,total_tokens,decoded
0,8,260,6,8,the movie has beautiful scenes but
1,16,260,14,16,the movie has beautiful scenes but an unconvin...
2,32,260,30,32,the movie has beautiful scenes but an unconvin...
3,64,260,62,64,the movie has beautiful scenes but an unconvin...
4,128,260,126,128,the movie has beautiful scenes but an unconvin...
5,256,260,254,256,the movie has beautiful scenes but an unconvin...


## 7. Kiểm tra cuối và phần sinh viên tự làm

Các assertions kiểm tra shape, giới hạn IDs và mask padding. Bảng truncation đếm token theo tokenizer, không dùng số từ để giả định số tokens.

- `TODO(student)`: special tokens nào có attention mask bằng 1?
- `TODO(student)`: câu ngắn có cần padding nếu chạy một mình không?
- `TODO(student)`: nội dung nào mất đi khi max_length giảm?
- `TODO(student)`: phân biệt words, content pieces, continuation pieces và special tokens trong Bài 10.